In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

# ─── Colab 参数（按需改）──────────────────────────────────────
REPO_URL = "https://github.com/Beater-221E/llm4rec-bias-Integrated.git"
REPO_DIR = "/content/llm4rec-bias-Integrated"
BRANCH = "main"                 # 或 master / 你的分支名
GITHUB_TOKEN = ""               # 私有仓库填 PAT；公有留空
MOUNT_DRIVE = False             # True → 挂载 Google Drive（可选缓存）
INSTALL_DEEPSPEED = False       # 0.5B 单卡不需要
# ─────────────────────────────────────────────────────────────

def _in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return Path("/content").exists() and "COLAB_RELEASE_TAG" in os.environ

IN_COLAB = _in_colab()
print(f"Colab = {IN_COLAB}")

if not IN_COLAB:
    print("非 Colab 环境，跳过本 cell。请直接跑下一个「实验参数」cell。")
else:
    try:
        import torch
        if torch.cuda.is_available():
            name = torch.cuda.get_device_name(0)
            mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
            print(f"GPU: {name}  ({mem:.1f} GB)")
        else:
            print("WARN: 未检测到 GPU。Runtime → Change runtime type → A100/GPU")
    except ImportError:
        print("torch 尚未导入，稍后随 requirements 安装/校验")

    if MOUNT_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        print("Drive mounted → /content/drive")

    repo = Path(REPO_DIR)
    url = REPO_URL
    if GITHUB_TOKEN.strip():
        url = url.replace("https://", f"https://{GITHUB_TOKEN.strip()}@")

    if (repo / "src" / "llm4rec").exists() and (repo / "pyproject.toml").exists():
        print(f"仓库已存在: {repo}")
        try:
            subprocess.check_call(["git", "-C", str(repo), "fetch", "--depth", "1", "origin", BRANCH])
            subprocess.check_call(["git", "-C", str(repo), "checkout", BRANCH])
            subprocess.check_call(["git", "-C", str(repo), "pull", "--ff-only", "origin", BRANCH])
            print("已更新到最新")
        except subprocess.CalledProcessError as e:
            print(f"git pull 跳过（可忽略）: {e}")
    else:
        if repo.exists():
            print(f"目录存在但不完整，删除后重 clone: {repo}")
            subprocess.check_call(["rm", "-rf", str(repo)])
        print(f"git clone → {repo}")
        subprocess.check_call(
            ["git", "clone", "--depth", "1", "--branch", BRANCH, url, str(repo)]
        )

    os.chdir(repo)
    os.environ["LLM4REC_ROOT"] = str(repo)
    src = str(repo / "src")
    if src not in sys.path:
        sys.path.insert(0, src)
    os.environ["PYTHONPATH"] = src + (
        ":" + os.environ["PYTHONPATH"] if os.environ.get("PYTHONPATH") else ""
    )

    print("安装依赖（requirements + editable）…")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "pip", "-q"])
    # Colab 通常已有较新 torch/CUDA，避免被 requirements 里的 torch 约束重装搞坏
    subprocess.check_call(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "omegaconf>=2.3", "pyyaml>=6.0", "rich>=13.0", "tqdm>=4.66",
            "numpy>=1.26", "pandas>=2.0", "scikit-learn>=1.4",
            "transformers>=4.45", "accelerate>=0.34", "datasets>=2.20",
            "sentence-transformers>=3.0", "wandb>=0.17",
        ]
    )
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", str(repo), "-q"])
    if INSTALL_DEEPSPEED:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "deepspeed", "-q"])

    for key in list(sys.modules):
        if key == "llm4rec" or key.startswith("llm4rec."):
            del sys.modules[key]
    import llm4rec
    import torch
    path = list(llm4rec.__path__)[0]
    print(f"ROOT     = {repo}")
    print(f"llm4rec  = {path}")
    print(f"torch    = {torch.__version__}  cuda={torch.cuda.is_available()}")
    assert str(repo) in path or path.startswith(str(repo / "src")), path
    print("Colab 环境就绪 ✓  → 继续跑下一个 cell")


In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path

# ─── A100 单卡 × Qwen2.5-0.5B-Instruct ───────────────────────
EXP = "minionerec_qwen05b_amazon"
SMOKE = False
GPUS = "0"
WANDB_MODE = "disabled"
FORCE_PREPARE = True               # SID 重建完可改回 False
DEEPSPEED = ""                      # 0.5B 单卡不用 ZeRO
OVERRIDES: list[str] = [
    "hardware.precision=bf16",       # A100 用 bf16；V100 改回 fp32
    "hardware.devices=0",
    # SID：接受碰撞、保持 3 层；只要求码本不塌缩
    "sid.collision_handling=none",
    "sid.max_collision_rate=1.0",
    "sid.require_unique=false",
    "sid.min_codebook_usage=0.5",    # 每层至少用一半码本，否则视为塌缩失败
]
# ─────────────────────────────────────────────────────────────

def _find_root() -> Path:
    env = os.environ.get("LLM4REC_ROOT")
    candidates = []
    if env:
        candidates.append(Path(env))
    candidates += [
        Path.cwd(),
        Path.cwd().parent,
        Path("/content/llm4rec-bias-Integrated"),
        Path("/home/sheng/proj/llm4rec-bias-Integrated"),
    ]
    content = Path("/content")
    if content.is_dir():
        candidates += sorted(p for p in content.iterdir() if p.is_dir())[:40]
    seen: set[Path] = set()
    for c in candidates:
        if not c.exists():
            continue
        c = c.resolve()
        if c in seen:
            continue
        seen.add(c)
        if (c / "src" / "llm4rec").exists() and (c / "pyproject.toml").exists():
            return c
    raise FileNotFoundError(
        "找不到项目根目录。\n"
        "Colab：请先运行最上方「Colab 专用」cell。\n"
        "本地：在仓库根目录打开本 notebook，或设置 LLM4REC_ROOT。"
    )

ROOT = _find_root()
os.chdir(ROOT)
src = str(ROOT / "src")
sys.path = [p for p in sys.path if "llm4rec" not in p or p == src]
sys.path.insert(0, src)
os.environ["LLM4REC_ROOT"] = str(ROOT)
os.environ["PYTHONPATH"] = src + (
    ":" + os.environ["PYTHONPATH"] if os.environ.get("PYTHONPATH") else ""
)

if SMOKE and not EXP.startswith("smoke_"):
    EXP = f"smoke_{EXP.split('_')[0]}"
    print(f"[SMOKE] EXP → {EXP}")

os.environ["CUDA_VISIBLE_DEVICES"] = GPUS
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["WANDB_MODE"] = WANDB_MODE
os.environ.setdefault("WANDB_PROJECT", "llm4rec-bias")

n_gpu = len([g for g in GPUS.split(",") if g.strip()])
if n_gpu > 1:
    os.environ.setdefault("NCCL_P2P_DISABLE", "1")
    os.environ.setdefault("NCCL_IB_DISABLE", "1")

print(f"ROOT     = {ROOT}")
print(f"EXP      = {EXP}")
print(f"GPUS     = {GPUS}  (n={n_gpu})")
print(f"WANDB    = {WANDB_MODE}")
print(f"DEEPSPEED= {DEEPSPEED or '<none>'}")
print(f"OVERRIDES= {OVERRIDES or '<none>'}")


In [ ]:
# 环境自检（Colab 上依赖应已由首个 cell 装好；本地则在此补装）
import torch

print(f"Python   : {sys.version.split()[0]}")
print(f"torch    : {torch.__version__}")
print(f"CUDA avail: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"  GPU{i}: {props.name}  {props.total_memory / 1024**3:.1f} GB")

def _ensure_llm4rec() -> None:
    for key in list(sys.modules):
        if key == "llm4rec" or key.startswith("llm4rec."):
            del sys.modules[key]
    try:
        import llm4rec
        path = getattr(llm4rec, "__path__", None)
        path0 = list(path)[0] if path else str(getattr(llm4rec, "__file__", ""))
        if str(ROOT) in path0 or path0.startswith(str(ROOT / "src")):
            print(f"llm4rec  : OK  ({path0})")
            return
        print(f"llm4rec 指向了别的路径: {path0}，改为安装本仓库…")
    except ImportError:
        print("llm4rec 未安装，正在 pip install -e . …")

    subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", str(ROOT), "-q"])
    req = ROOT / "requirements.txt"
    if req.exists():
        # 不强制重装 torch，避免弄坏 Colab CUDA 轮子
        subprocess.check_call(
            [
                sys.executable, "-m", "pip", "install", "-q",
                "omegaconf", "pyyaml", "rich", "tqdm", "numpy", "pandas",
                "scikit-learn", "transformers", "accelerate", "datasets",
                "sentence-transformers", "wandb",
            ]
        )
    for key in list(sys.modules):
        if key == "llm4rec" or key.startswith("llm4rec."):
            del sys.modules[key]
    import llm4rec
    path0 = list(llm4rec.__path__)[0]
    print(f"llm4rec  : OK  ({path0})")
    if str(ROOT) not in path0 and not path0.startswith(str(ROOT / "src")):
        raise RuntimeError(f"llm4rec 仍不指向本仓库: {path0}")

_ensure_llm4rec()


In [ ]:
from llm4rec.cli.main import cmd_list

cmd_list()


In [ ]:
from llm4rec.core.compose import compose, to_dict, validate
from llm4rec.cli.main import _print_plan

cli_overrides = list(OVERRIDES)
if DEEPSPEED:
    cli_overrides.append(f"hardware.deepspeed={DEEPSPEED}")

cfg = validate(to_dict(compose(EXP, cli_overrides)))
_print_plan(cfg)
print("\n配置校验通过 ✓")

ROUTE = cfg["experiment"]["route"]
NEEDS_SID = "sid" in cfg
NEEDS_BM25 = str((cfg.get("decoder") or {}).get("name")) == "bm25_query"
print(f"\nroute={ROUTE}  needs_sid={NEEDS_SID}  needs_bm25={NEEDS_BM25}")
print(f"stages = {' → '.join(cfg['stages'])}")


In [ ]:
from llm4rec.cli.main import cmd_download_data

cmd_download_data(cfg, force=FORCE_PREPARE)


In [ ]:
from llm4rec.cli.main import cmd_prepare_data
from llm4rec.data.base import get_adapter

cmd_prepare_data(cfg, force=FORCE_PREPARE)

adapter = get_adapter(cfg)
proc_dir = adapter.processed_dir(cfg)
print(f"\nprocessed dir: {proc_dir}")
for name in ("interactions.jsonl", "item_meta.json", "popularity.json", "stats.json"):
    p = Path(proc_dir) / name
    print(f"  {'✓' if p.exists() else '✗'} {name}  ({p.stat().st_size if p.exists() else 0} bytes)")


In [ ]:
# 快速看一眼数据规模与划分
stats_path = Path(proc_dir) / "stats.json"
if stats_path.exists():
    stats = json.loads(stats_path.read_text())
    print(json.dumps(stats, indent=2, ensure_ascii=False)[:2000])
else:
    print("stats.json 不存在")


In [ ]:
from llm4rec.cli.main import cmd_embed_items

if NEEDS_SID:
    cmd_embed_items(cfg, force=FORCE_PREPARE)
else:
    print(f"[embed] route={ROUTE} 不需要 SID embedding，跳过")


In [ ]:
from llm4rec.cli.main import cmd_build_sid
import shutil

# 当前 OVERRIDES：不强制 collision=0，只检查码本利用率（min_codebook_usage）
# 若上次 build 失败留下空目录，先清掉再重建
if NEEDS_SID:
    broken = list((ROOT / "artifacts" / "sid").glob("*/*/"))
    for d in broken:
        if d.is_dir() and not (d / "manifest.json").exists():
            print(f"[sid] 删除残缺目录: {d}")
            shutil.rmtree(d, ignore_errors=True)
    # 必须 True 才会覆盖；残缺目录上面已删，这里即使 False 也会新建
    cmd_build_sid(cfg, force=FORCE_PREPARE)
else:
    print(f"[sid] route={ROUTE} 不用 SID，跳过")


In [ ]:
from llm4rec.cli.main import cmd_build_bm25

if NEEDS_BM25:
    cmd_build_bm25(cfg, force=FORCE_PREPARE)
else:
    print(f"[bm25] route={ROUTE} 不用 BM25，跳过")


In [ ]:
def _tree(path: Path, depth: int = 2, prefix: str = "") -> None:
    if not path.exists():
        print(f"{prefix}✗ {path} (missing)")
        return
    print(f"{prefix}{path.name}/" if path.is_dir() else f"{prefix}{path.name}")
    if path.is_dir() and depth > 0:
        kids = sorted(path.iterdir())[:30]
        for k in kids:
            _tree(k, depth - 1, prefix + "  ")

print("=== processed ===")
_tree(Path(proc_dir), depth=1)

if NEEDS_SID:
    sid_root = ROOT / "artifacts" / "sid"
    print("\n=== sid ===")
    _tree(sid_root, depth=3)

if NEEDS_BM25:
    bm25_root = ROOT / "artifacts" / "bm25"
    print("\n=== bm25 ===")
    _tree(bm25_root, depth=2)

print("\n数据准备完成 ✓")


In [ ]:
# 组装与 run.sh 等价的启动命令（多卡 → torchrun）
import shlex
import subprocess

STAGES = ""          # 留空 = 用配置默认；也可改成 "sft,eval" 只跑前半段
RESUME_FROM = ""     # 例: "runs/.../sft/final"；留空从 backbone 开始

args = ["--config", EXP]
if STAGES:
    args += ["--stages", STAGES]
if RESUME_FROM:
    args += ["--resume-from", RESUME_FROM]
if DEEPSPEED:
    args += [f"hardware.deepspeed={DEEPSPEED}"]
args += list(OVERRIDES)

env = os.environ.copy()
env["PYTHONPATH"] = str(ROOT / "src") + (":" + env["PYTHONPATH"] if env.get("PYTHONPATH") else "")

if n_gpu > 1:
    import random
    master_port = 20000 + random.randint(0, 19999)
    cmd = [
        "torchrun", "--standalone",
        "--nproc_per_node", str(n_gpu),
        "--master_port", str(master_port),
        "-m", "llm4rec.cli.main", "run", *args,
    ]
else:
    cmd = [sys.executable, "-m", "llm4rec.cli.main", "run", *args]

print("即将执行:")
print(" ".join(shlex.quote(c) for c in cmd))
print(f"\nstages = {STAGES or cfg['stages']}")
print(f"SMOKE  = {SMOKE}   （正式实验请设 SMOKE=False 并选非 smoke EXP）")


In [ ]:
# ★ 真正开训。耗时取决于 SMOKE / 模型大小 / 卡数。
# 日志同时写入 logs/；也可在终端用 bash run.sh 跑同一套参数。

logs_dir = ROOT / "logs"
logs_dir.mkdir(exist_ok=True)
from datetime import datetime

log_path = logs_dir / f"{EXP}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
print(f"log → {log_path}")

with open(log_path, "w") as logf:
    proc = subprocess.Popen(
        cmd,
        cwd=str(ROOT),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
        logf.write(line)
    rc = proc.wait()

print(f"\nexit code = {rc}")
if rc != 0:
    raise RuntimeError(f"训练失败，见日志: {log_path}")


In [ ]:
# 定位本次（或最近一次）run 目录
from llm4rec.data.base import get_adapter

adapter = get_adapter(cfg)
dataset_key = adapter.dataset_key(cfg)
model_name = str(cfg["model"]["name"]).replace("/", "_")
seed = cfg["seed"]

run_root = ROOT / "runs" / dataset_key / ROUTE / model_name / f"seed_{seed}"
if not run_root.exists():
    candidates = sorted((ROOT / "runs").rglob("summary.json"))
    if not candidates:
        raise FileNotFoundError("找不到任何 runs/*/summary.json，请先完成训练 cell")
    RUN_DIR = candidates[-1].parent
else:
    stamps = sorted([p for p in run_root.iterdir() if p.is_dir()])
    if not stamps:
        raise FileNotFoundError(f"{run_root} 下没有 run")
    RUN_DIR = stamps[-1]

print(f"RUN_DIR = {RUN_DIR}")
for name in ("resolved_config.json", "summary.json", "metrics.jsonl", "eval"):
    p = RUN_DIR / name
    print(f"  {'✓' if p.exists() else '✗'} {name}")


In [ ]:
summary = json.loads((RUN_DIR / "summary.json").read_text())
print("=== summary.json (checkpoints) ===")
for stage, payload in summary.items():
    if isinstance(payload, dict):
        ckpt = payload.get("checkpoint")
        keys = [k for k in payload if k != "checkpoint"][:8]
        print(f"\n[{stage}]")
        if ckpt:
            print(f"  checkpoint: {ckpt}")
        for k in keys:
            v = payload[k]
            if isinstance(v, float):
                print(f"  {k}: {v:.6f}")
            elif isinstance(v, (int, str, bool)) or v is None:
                print(f"  {k}: {v}")


In [ ]:
eval_dir = RUN_DIR / "eval"
eval_files = sorted(eval_dir.glob("eval_*.json")) if eval_dir.exists() else []
delta_path = eval_dir / "bias_delta.json"

print(f"找到 {len(eval_files)} 次 stage-end eval\n")

KEY_METRICS = [
    "hr@10", "ndcg@10", "hr_ips@10", "ndcg_ips@10",
    "pop_lift@1", "pop_lift@10", "delta_gap",
    "exposure_gini", "coverage@10", "tier_gap",
    "history_copy_rate", "top1_concentration", "valid_rate",
]


def _metrics_dict(payload: dict) -> dict:
    """eval_*.json → payload['metrics']；bias_delta 本身是扁平 dict（键带 delta/）。"""
    if isinstance(payload.get("metrics"), dict):
        return payload["metrics"]
    return payload


def _pick(flat: dict, keys: list[str]) -> dict:
    return {k: flat[k] for k in keys if k in flat}


rows = []
for ef in eval_files:
    payload = json.loads(ef.read_text())
    flat = _metrics_dict(payload)
    picked = _pick(flat, KEY_METRICS)
    rows.append((ef.stem, picked))
    ckpt = payload.get("checkpoint")
    print(f"── {ef.name} ──" + (f"  ckpt={ckpt}" if ckpt else ""))
    for k, v in picked.items():
        print(f"  {k:24s} {v:.6f}" if isinstance(v, float) else f"  {k:24s} {v}")
    print()

delta = None
if delta_path.exists():
    delta = json.loads(delta_path.read_text())
    print("═══ bias_delta.json (last − first) ═══")
    print("    键名带 delta/ 前缀（由 bias_delta() 生成）\n")
    shown = set()
    for k in KEY_METRICS:
        dk = f"delta/{k}"
        if dk in delta:
            v = delta[dk]
            arrow = "↑" if v > 0 else ("↓" if v < 0 else "·")
            print(f"  {dk:28s} {v:+.6f}  {arrow}")
            shown.add(dk)
    extras = [k for k, v in delta.items() if k not in shown and isinstance(v, (int, float))]
    if extras:
        print("\n  (其他 delta 字段)")
        for k in extras[:30]:
            v = delta[k]
            print(f"  {k:28s} {v:+.6f}" if isinstance(v, float) else f"  {k:28s} {v}")
else:
    print("尚无 bias_delta.json（需要至少两次 stage-end eval）")


In [ ]:
# 可选：把关键指标画成简单对比表（SFT vs RL/DPO）
try:
    import pandas as pd
    from IPython.display import display

    if len(rows) >= 1:
        table = {name: picked for name, picked in rows}
        if delta is not None:
            table["bias_delta"] = {
                k: delta.get(f"delta/{k}") for k in KEY_METRICS if f"delta/{k}" in delta
            }
        df = pd.DataFrame(table).T
        df = df.apply(pd.to_numeric, errors="coerce")
        display(df.round(6))
    else:
        print("没有 eval 结果可展示")
except Exception as e:
    print(f"表格展示跳过: {e}")


In [ ]:
# 浏览 metrics.jsonl 里的在线 bias 曲线（RL/DPO 训练中每 N step）
metrics_path = RUN_DIR / "metrics.jsonl"
if not metrics_path.exists():
    print("无 metrics.jsonl")
else:
    online = []
    with metrics_path.open() as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
            except json.JSONDecodeError:
                continue
            keys = rec.keys() if isinstance(rec, dict) else []
            if any(str(k).startswith("bias/") or str(k).startswith("bias.") for k in keys):
                online.append(rec)
            elif isinstance(rec, dict) and "bias" in rec:
                online.append(rec)

    print(f"metrics.jsonl 含 bias 字段的记录: {len(online)}")
    if online:
        print("最近 3 条:")
        for rec in online[-3:]:
            brief = {k: v for k, v in rec.items() if not str(k).startswith("_")}
            print(json.dumps(brief, ensure_ascii=False)[:500])

    env_path = RUN_DIR / "environment.json"
    if env_path.exists():
        print(f"\nenvironment.json → {env_path}")


In [ ]:
# 填一个已有 checkpoint 目录；留空则跳过
EVAL_ONLY_CKPT = ""  # 例: str(RUN_DIR / "rl" / "final")
EVAL_OVERRIDES = [
    # "bias.final_examples=256",
    # "evaluation.top_k=[1,5,10,20]",
]


In [ ]:
if not EVAL_ONLY_CKPT:
    print("EVAL_ONLY_CKPT 为空，跳过单独评测。")
else:
    ckpt = Path(EVAL_ONLY_CKPT)
    if not ckpt.exists():
        raise FileNotFoundError(ckpt)
    eval_args = ["--config", EXP, "--stages", "eval", "--resume-from", str(ckpt), *EVAL_OVERRIDES, *OVERRIDES]
    if n_gpu > 1:
        import random
        master_port = 20000 + random.randint(0, 19999)
        eval_cmd = [
            "torchrun", "--standalone",
            "--nproc_per_node", str(n_gpu),
            "--master_port", str(master_port),
            "-m", "llm4rec.cli.main", "run", *eval_args,
        ]
    else:
        eval_cmd = [sys.executable, "-m", "llm4rec.cli.main", "run", *eval_args]

    print("执行:", " ".join(shlex.quote(c) for c in eval_cmd))
    proc = subprocess.run(eval_cmd, cwd=str(ROOT), env=env)
    if proc.returncode != 0:
        raise RuntimeError(f"eval-only 失败, exit={proc.returncode}")
    print("单独评测完成 ✓")
